In [2]:
%pip install pandas
import pandas as pd
import boto3
import json
import gzip
from io import StringIO, BytesIO


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [3]:
bucket = "s3-3601-final-project"
file_key = "ncdot/traffic/aadt_stations.json.gz"


s3_client = boto3.client("s3")

response = s3_client.get_object(Bucket=bucket, Key=file_key)
compressed_content = response["Body"].read()

with gzip.GzipFile(fileobj=BytesIO(compressed_content)) as gzip_file:
    raw_json_data = json.loads(gzip_file.read().decode("utf-8"))

df = pd.json_normalize(raw_json_data, record_path="features")
df.columns = [col.replace("attributes.", "") for col in df.columns]
df.head(10)

,LocationID,ROUTE,LOCATION,AADT_2022,geometry.x,geometry.y
0,0600000063,NC 49,WEST OF I-485,32500,1.489478e+06,572619.455661
1,0600000145,NC 49,EAST OF SR 5394 MALLARD CK CH RD,42000,1.487602e+06,571943.155568
2,0600000829,NC 49,EAST OF SR 2939,32500,1.485991e+06,570949.346742
3,0600000107,NC 49,EAST OF SUTHER RD,26500,1.484530e+06,569785.046649
4,0600000108,NC 49,WEST OF SUTHER RD,27500,1.483872e+06,569289.051488
5,0600000414,NC 24 (E WT HARRIS BLVD),EAST OF SR 5316,51500,1.480853e+06,566180.543380
6,0600000412,NC 24 (WT HARRIS BLVD),SOUTH OF US 29,47500,1.478411e+06,570221.959810
7,0600000044,US 29/NC 49,EAST OF STETSON DR,29500,1.474954e+06,564389.520486
8,0600000065,US 29,SOUTH OF UNIVERSITY POINTE BLVD,18000,1.476252e+06,566715.883343
9,0600001182,SR 2833,NORTH OF NC 49,15500,1.485507e+06,571041.096891


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   LocationID  10 non-null     object 
 1   ROUTE       10 non-null     object 
 2   LOCATION    10 non-null     object 
 3   AADT_2022   10 non-null     object 
 4   geometry.x  10 non-null     float64
 5   geometry.y  10 non-null     float64
dtypes: float64(2), object(4)
memory usage: 612.0+ bytes


In [6]:
import requests

url = "https://services.arcgis.com/NuWFvHYDMVmmxMeM/ArcGIS/rest/services/NCDOT_AADT_Stations/FeatureServer/0/query"

location_ids = [
   "0600000063",
   "0600000145",
   "0600000829",
   "0600000107",
   "0600000108",
   "0600000414", 
   "0600000412",
   "0600000044",
   "0600000065",
   "0600001182",
 
]

where = "LocationID IN ({})".format(
    ",".join(f"'{x}'" for x in location_ids)
)

params = {
    "where": where,
    "outFields": "LocationID,COUNTY,ROUTE,LOCATION," + ",".join([f"AADT_{year}" for year in range(2018, 2023)]),
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "json"
}

response = requests.get(url, params=params)
data = response.json()

df = pd.json_normalize(data["features"])

df = df.rename(columns={
    "geometry.x": "longitude",
    "geometry.y": "latitude"
})

df.head(10)

,attributes.LocationID,attributes.COUNTY,attributes.ROUTE,attributes.LOCATION,attributes.AADT_2018,attributes.AADT_2019,attributes.AADT_2020,attributes.AADT_2021,attributes.AADT_2022,longitude,latitude
0,0600000044,MECKLENBURG,US 29/NC 49,EAST OF STETSON DR,29500,,27000,,29500,-80.759574,35.287985
1,0600000063,MECKLENBURG,NC 49,WEST OF I-485,32000,,,,32500,-80.711385,35.311292
2,0600000065,MECKLENBURG,US 29,SOUTH OF UNIVERSITY POINTE BLVD,18500,,17000,,18000,-80.755360,35.294440
3,0600000107,MECKLENBURG,NC 49,EAST OF SUTHER RD,26000,,,,26500,-80.727801,35.303270
4,0600000108,MECKLENBURG,NC 49,WEST OF SUTHER RD,26000,,,,27500,-80.729977,35.301876
5,0600000145,MECKLENBURG,NC 49,EAST OF SR 5394 MALLARD CK CH RD,38500,,,,42000,-80.717633,35.309346
6,0600000412,MECKLENBURG,NC 24 (WT HARRIS BLVD),SOUTH OF US 29,50000,,41500,,47500,-80.748333,35.304176
7,0600000414,MECKLENBURG,NC 24 (E WT HARRIS BLVD),EAST OF SR 5316,,,45000,,51500,-80.739913,35.293191
8,0600000829,MECKLENBURG,NC 49,EAST OF SR 2939,,,28500,,32500,-80.722975,35.306539
9,0600001182,MECKLENBURG,SR 2833,NORTH OF NC 49,,,,,15500,-80.724601,35.306768


In [7]:
df.isnull().sum()

attributes.LocationID    0
attributes.COUNTY        0
attributes.ROUTE         0
attributes.LOCATION      0
attributes.AADT_2018     0
attributes.AADT_2019     0
attributes.AADT_2020     0
attributes.AADT_2021     0
attributes.AADT_2022     0
longitude                0
latitude                 0
dtype: int64

In [ ]:
import matplotlib.pyplot as plt

year_cols = [f"attributes.AADT_{year}" for year in range(2018, 2023)]
aadt = df[year_cols].apply(pd.to_numeric, errors="coerce")
aadt.columns = [col.rsplit("_", 1)[1] for col in year_cols]
aadt.index = df["attributes.LocationID"]

ax = aadt.T.plot(marker="o", figsize=(11, 6))
ax.set(
    title="Annual AADT (Traffic Volume) by Station",
    xlabel="Year",
    ylabel="Average vehicles per day",
)
ax.legend(title="Location ID", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()